# 02 统计计算

问题与建模思路参考 Allen B. Downey *Think Bayes*（中译《贝叶斯思维》）第 2 章。

**学习目标**：
- 用 `Pmf` 表示离散概率分布并完成归一化
- 用 `Suite` 封装「先验 → 似然 → 后验」循环
- 把第 1 章的曲奇饼与 Monty Hall 重写成可扩展框架


## 1. 分布与 Pmf

离散随机变量 $X$ 的概率质量函数：

$$
p(x) = P(X=x),\qquad \sum_x p(x)=1
$$

`thinkbayes_mini.Pmf` 用 `dict` 存未归一或已归一的质量，`Normalize()` 令总和为 1。


In [1]:
import sys
from pathlib import Path

# 保证可从 notebooks/bayes 或仓库根目录运行
HERE = Path.cwd()
for candidate in [HERE, HERE / "notebooks" / "bayes", HERE.parent]:
    if (candidate / "thinkbayes_mini.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break

from thinkbayes_mini import Pmf, Suite

pmf = Pmf()
for x in [1, 2, 3, 4, 5, 6]:
    pmf.Set(x, 1)
pmf.Normalize()
print("公平骰子:", dict(pmf.Items()))
print("P(6) =", pmf.Prob(6))


公平骰子: {1: 0.16666666666666666, 2: 0.16666666666666666, 3: 0.16666666666666666, 4: 0.16666666666666666, 5: 0.16666666666666666, 6: 0.16666666666666666}
P(6) = 0.16666666666666666


## 2. 贝叶斯框架：Suite

对每个假设 $h$：

$$
p(h) \leftarrow p(h)\,P(D \mid h)
\quad\text{然后}\quad
p(h) \leftarrow \frac{p(h)}{\sum_{h'} p(h')}
$$

子类实现 `Likelihood(data, hypo)`，调用 `Update(data)` 即可。


In [2]:
class Cookie(Suite):
    """两碗曲奇饼：hypo 为碗名，data 为味道字符串。"""

    mixes = {
        "Bowl1": dict(vanilla=0.75, chocolate=0.25),
        "Bowl2": dict(vanilla=0.5, chocolate=0.5),
    }

    def Likelihood(self, data, hypo):
        return self.mixes[hypo][data]


suite = Cookie(["Bowl1", "Bowl2"])
suite.Normalize()
print("先验:", dict(suite.Items()))
suite.Update("vanilla")
print("后验(香草):", {h: round(p, 4) for h, p in suite.Items()})
assert abs(suite.Prob("Bowl1") - 0.6) < 1e-9


先验: {'Bowl1': 0.5, 'Bowl2': 0.5}
后验(香草): {'Bowl1': 0.6, 'Bowl2': 0.4}


## 3. Monty Hall 封装

假设：你先选门 1；`data` 为 Monty 打开的门号。


In [3]:
class Monty(Suite):
    def Likelihood(self, data, hypo):
        """data: Monty 打开的门；hypo: 奖品所在门；玩家始终选门 1。"""
        if hypo == data:
            return 0  # Monty 不会打开有奖的门
        if hypo == 1:
            return 0.5  # 奖在门 1，Monty 在 2/3 中随机开
        return 1  # 奖在另一扇未选门，Monty 只能开剩下那扇


monty = Monty([1, 2, 3])
monty.Normalize()
monty.Update(3)  # Monty 开门 3
print({h: round(p, 4) for h, p in monty.Items()})
assert abs(monty.Prob(2) - 2 / 3) < 1e-9


{1: 0.3333, 2: 0.6667, 3: 0.0}


## 4. M&M 豆（Suite 版）

假设编码为元组 `(bag_yellow, bag_green)` 的年份。


In [4]:
mix94 = dict(brown=30, yellow=20, red=20, green=10, orange=10, tan=10)
mix96 = dict(blue=24, green=20, orange=16, yellow=14, red=13, brown=13)


class MAndM(Suite):
    mixes = {1994: mix94, 1996: mix96}

    def Likelihood(self, data, hypo):
        """data: 颜色；hypo: (黄豆来自哪年袋, 绿豆来自哪年袋)。"""
        color = data
        bag_year = hypo[0] if color == "yellow" else hypo[1]
        return self.mixes[bag_year][color] / 100


# A: 黄来自94、绿来自96；B: 相反
suite_mm = MAndM([(1994, 1996), (1996, 1994)])
suite_mm.Normalize()
suite_mm.Update("yellow")
suite_mm.Update("green")
print("M&M 后验:", {h: round(p, 4) for h, p in suite_mm.Items()})


M&M 后验: {(1994, 1996): 0.7407, (1996, 1994): 0.2593}


## 5. 框架带来的好处

- **似然局部化**：问题相关逻辑集中在 `Likelihood`，更新循环不变。
- **多次数据**：`UpdateSet` 或连续 `Update` 自动连乘似然。
- **可组合**：后验可再当先验，接到估计、决策等章节。


In [5]:
cookie = Cookie(["Bowl1", "Bowl2"])
cookie.Normalize()
cookie.UpdateSet(["vanilla", "chocolate", "vanilla"])
print("三次抽取后:", {h: round(p, 4) for h, p in cookie.Items()})


三次抽取后: {'Bowl1': 0.5294, 'Bowl2': 0.4706}


## 小结

1. `Pmf` 管分布算术；`Suite` 管贝叶斯更新。
2. 写新问题 = 定假设空间 + 实现 `Likelihood`。
3. 下一章用同一框架做**参数估计**（骰子面数、火车头编号）。
